# 03 — Train / Validation / Test Split

## Objective

Split the labeled order-level dataset into training, validation, and test sets while preserving the chronological order of purchases.

A **time-based split** is used instead of a random split to better simulate a real-world prediction scenario, where the model is trained on historical orders and evaluated on future orders.

### Split Strategy

- **70% — Training Set**
- **15% — Validation Set**
- **15% — Test Set**

The test set is kept completely unseen until the final model evaluation.

## 1. Load the Labeled Dataset

Load the labeled order-level dataset created in Notebook 02 and inspect its shape and target distribution before performing the split.

This ensures that the target variable is available and provides a quick check of the dataset before applying the chronological splitting strategy.

In [1]:
import pandas as pd

df = pd.read_csv("labeled_table.csv")

print("Shape:", df.shape)

print("\nLabel distribution:")
print(df["is_late"].value_counts())

print("\nLabel percentages:")
print(df["is_late"].value_counts(normalize=True) * 100)

Shape: (96476, 24)

Label distribution:
is_late
0    88649
1     7827
Name: count, dtype: int64

Label percentages:
is_late
0    91.887101
1     8.112899
Name: proportion, dtype: float64


## 2. Inspect the Time Distribution

Examine the order purchase timeline and compare late-delivery rates across years before applying the chronological split.

This helps verify that delivery behavior changes over time and supports the decision to use a time-based split rather than a random split.

In [2]:
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
)

print("Start date:", df["order_purchase_timestamp"].min())
print("End date:", df["order_purchase_timestamp"].max())

print("\nOrders by year:")
print(df["order_purchase_timestamp"].dt.year.value_counts().sort_index())

print("\nLate rate by year:")
print(
    df.groupby(df["order_purchase_timestamp"].dt.year)["is_late"]
      .mean()
      .mul(100)
)

Start date: 2016-09-15 12:16:38
End date: 2018-08-29 15:00:37

Orders by year:
order_purchase_timestamp
2016      272
2017    43426
2018    52778
Name: count, dtype: int64

Late rate by year:
order_purchase_timestamp
2016    1.470588
2017    6.627366
2018    9.369434
Name: is_late, dtype: float64


### Finding

The late-delivery rate changes noticeably over time, increasing from **1.47% in 2016** to **6.63% in 2017** and **9.37% in 2018**.

Because delivery behavior varies across time, a **time-based split** is more appropriate than a random split. This preserves the chronological order of the data and better reflects a real-world scenario in which a model is trained on historical orders and evaluated on future orders.

## 3. Create the Time-Based Split

Sort the labeled dataset chronologically by `order_purchase_timestamp` and divide it into:

- **70% — Training set**
- **15% — Validation set**
- **15% — Test set**

The oldest orders are assigned to the training set, followed by validation orders, while the most recent orders are reserved for the final test set.

This prevents future observations from being used to train a model that is intended to predict future delivery outcomes.

In [3]:
# Sort data chronologically
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

# Time-based split: 70% Train, 15% Validation, 15% Test
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain dates:")
print(train_df["order_purchase_timestamp"].min(), "->",
      train_df["order_purchase_timestamp"].max())

print("\nValidation dates:")
print(val_df["order_purchase_timestamp"].min(), "->",
      val_df["order_purchase_timestamp"].max())

print("\nTest dates:")
print(test_df["order_purchase_timestamp"].min(), "->",
      test_df["order_purchase_timestamp"].max())

Train shape: (67533, 24)
Validation shape: (14471, 24)
Test shape: (14472, 24)

Train dates:
2016-09-15 12:16:38 -> 2018-04-15 20:07:56

Validation dates:
2018-04-15 20:10:23 -> 2018-06-21 07:50:39

Test dates:
2018-06-21 08:29:29 -> 2018-08-29 15:00:37


## 4. Validate the Split

Verify the size and late-delivery rate of each dataset after the chronological split.

Because a time-based split preserves the natural temporal distribution of the data, the target proportions are not expected to be identical across the three sets.

In [4]:
for name, split in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    late_rate = split["is_late"].mean() * 100
    
    print(
        f"{name}: "
        f"{len(split)} rows | "
        f"Late = {late_rate:.2f}%"
    )

Train: 67533 rows | Late = 9.03%
Validation: 14471 rows | Late = 5.34%
Test: 14472 rows | Late = 6.61%


## 5. Save the Split Datasets

Save the training, validation, and test datasets as separate artifacts for the next stages of the machine-learning pipeline.

- `train.csv` — used for EDA, feature engineering, and model training
- `validation.csv` — used for model selection and tuning
- `test.csv` — reserved for the final model evaluation

In [5]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Train, validation, and test files saved successfully!")

Train, validation, and test files saved successfully!


## Conclusion

The labeled dataset was successfully divided using a chronological **70% / 15% / 15%** split.

- **Training set:** 67,533 orders
- **Validation set:** 14,471 orders
- **Test set:** 14,472 orders
- Chronological order was preserved to prevent future information from leaking into model training.
- The late-delivery rate varies across the three periods, which is expected in a time-based split.
- The three datasets were saved for use in the subsequent stages of the project.

The test set will remain unseen until the final model evaluation.